### Primitivas

In [2]:
(define esquemas '())

; primitiva para acessar campos de um registro
(define (acessa campo registro)
  (let ((esq (cdr (assq (car registro) esquemas))))
    (list-ref registro (cdr (assq campo esq)))))

; monta um esquema a partir de pares (campo . valor):
(define (construir tipo dados)
  (let* ((esq    (cdr (assq tipo esquemas)))
         (campos (map car (cdr esq))))
    (for-each
     (lambda (c)
       (unless (assq c dados)
         (error "maplexic cria. campo faltante:" c)))
     campos)
    (for-each
     (lambda (par)
       (unless (memq (car par) campos)
         (error "maplexic cria. campo desconhecido:" (car par))))
     dados)
    (cons tipo (map (lambda (c) (cdr (assq c dados))) campos))))

; devolve um novo registro com um campo trocado
(define (adiciona-campo esquema campo valor)
  (let* ((tag (car esquema))
         (esq (cdr (assq tag esquemas)))
         (i   (cdr (assq campo esq))))
    (append (list-head esquema i)
            (list valor)
            (list-tail esquema (+ i 1)))))

(display "primitivas prontas")

primitivas prontas

### Texturas

In [3]:
; Texturas (nome -> hex) e validação de cor.
(eval-when (expand load eval)

  (define texturas
    '((areia . "#d8c79a") (floresta . "#1f5130") (grama . "#5bb04a")
      (agua . "#3f7fe0") (pedra . "#8a8f99") (terra . "#7a4b2b")
      (lava . "#e0552f") (neve . "#eef3fb")))

  ; reconhece "#rgb" ou "#rrggbb"
  (define (cor-hex? s)
    (and (string? s)
         (> (string-length s) 1)
         (char=? (string-ref s 0) #\#)
         (let ((corpo (substring s 1)))
           (and (or (= (string-length corpo) 3) (= (string-length corpo) 6))
                (and-map (lambda (ch)
                           (or (char-numeric? ch)
                               (memv (char-downcase ch) '(#\a #\b #\c #\d #\e #\f))))
                         (string->list corpo))))))

  ; textura -> hex; hex -> hex; senão erro de compilação
  (define (resolve-cor v stx)
    (cond ((cor-hex? v) v)
          ((assq v texturas) => cdr)
          (else (syntax-violation 'maplexic
                  (string-append "cor desconhecida (use textura ou hex \"#rrggbb\"): "
                                 (cond ((symbol? v) (symbol->string v))
                                       ((string? v) v)
                                       (else "valor invalido")))
                  stx)))))

(display "texturas prontas")

texturas prontas

### Macro maplexic

In [4]:
; Macro 'maplexic' com quatro casos:
(define-syntax maplexic
  (lambda (stx)
    (syntax-case stx (esquema cria encontra de adiciona em)

      ; cria esquema para registros
      ; (maplexic esquema objeto nome posicao largura altura cor)
      ((_ esquema tipo campo ...)
       (let* ((tipo-d (syntax->datum #'tipo))
              (campos (syntax->datum #'(campo ...)))
              (pares  (cons (cons 'tipo 0)
                            (map cons campos (iota (length campos) 1)))))
         (with-syntax ((esq   (datum->syntax #'tipo (symbol-append tipo-d '-esquema)))
                       (alist (datum->syntax #'tipo pares)))
           #'(begin
               (define esq 'alist)
               (set! esquemas (cons (cons 'tipo esq) esquemas))))))

      ; cria entidade baseado no esquema (tipo)
      ; (maplexic cria objeto nome torre posicao (10 20) largura 30 altura 40 cor pedra)
      ((_ cria tipo par ...)
       (let* ((flat  (syntax->datum #'(par ...)))
              (dados (let recorre ((xs flat) (acc '()))
                       (cond ((null? xs) (reverse acc))
                             ((null? (cdr xs))
                              (syntax-violation 'maplexic "cria: campo sem valor" stx))
                             (else (recorre (cddr xs)
                                            (cons (cons (car xs) (cadr xs)) acc))))))
              (dados (map (lambda (p)
                            (if (eq? (car p) 'cor)
                                (cons 'cor (resolve-cor (cdr p) stx))
                                p))
                          dados)))
         (with-syntax ((dados-lit (datum->syntax #'tipo dados)))
           #'(construir 'tipo 'dados-lit))))

      ; busca campo em entidade
      ; (maplexic encontra cor de amostra)
      ((_ encontra campo de reg)
       #'(acessa 'campo reg))

      ; adiciona entidade em mapa
      ; (maplexic adiciona (castelo heroi) em mapa-principal)
      ((_ adiciona (obj ...) em mapa)
       #'(adiciona-campo mapa 'objetos (list obj ...))))))

(display "macro maplexic pronta")

macro maplexic pronta

### Esquemas

In [5]:
(maplexic esquema objeto nome posicao largura altura cor)
(maplexic esquema mapa nome largura altura objetos)

(display "esquema objeto: ")(display objeto-esquema)
(newline)
(display "esquema mapa:   ")(display mapa-esquema)

esquema objeto: ((tipo . 0) (nome . 1) (posicao . 2) (largura . 3) (altura . 4) (cor . 5))
esquema mapa:   ((tipo . 0) (nome . 1) (largura . 2) (altura . 3) (objetos . 4))

### Testes da linguagem

In [6]:
(display "Criando mapa...\n")
(define mapa-principal (maplexic cria mapa nome mapa-principal largura 1000 altura 1000 objetos ()))
(display mapa-principal)
(newline)

(display "Lendo largura:\n")
(display (maplexic encontra largura de mapa-principal))
(display "\nLendo altura:\n")
(display (maplexic encontra altura de mapa-principal))

Criando mapa...
(mapa mapa-principal 1000 1000 ())
Lendo largura:
1000
Lendo altura:
1000

In [7]:
(display "Criando objetos com texturas...\n")

; terrenos: texturas nomeadas (resolvidas em tempo de compilação)
(define mata  (maplexic cria objeto posicao (0 0) nome mata largura 150 altura 1000 cor floresta))
(define lago  (maplexic cria objeto nome lago  posicao (250 650) largura 300 altura 250  cor agua))
(define praia (maplexic cria objeto nome praia posicao (250 550) cor areia largura 300 altura 100))

; construções e personagens
(define castelo  (maplexic cria objeto nome castelo posicao (650 600) largura 300 altura 300 cor pedra))
(define estabulo (maplexic cria objeto nome estabulo-cheio posicao (200 150) largura 180 altura 120 cor grama))
(define heroi    (maplexic cria objeto nome heroi posicao (560 580) largura 30 altura 40 cor "#ff552f"))
(define torre-mago (maplexic cria objeto cor "#8a4fd0" nome torre-mago largura 60 posicao (720 350) altura 200))

(for-each (lambda (o) (display o) (newline))
          (list mata lago praia castelo estabulo heroi torre-mago))

Criando objetos com texturas...
(objeto mata (0 0) 150 1000 #1f5130)
(objeto lago (250 650) 300 250 #3f7fe0)
(objeto praia (250 550) 300 100 #d8c79a)
(objeto castelo (650 600) 300 300 #8a8f99)
(objeto estabulo-cheio (200 150) 180 120 #5bb04a)
(objeto heroi (560 580) 30 40 #ff552f)
(objeto torre-mago (720 350) 60 200 #8a4fd0)


In [8]:
; Como fica uma criação de objeto depois de EXPANDIDA pela macro.
(use-modules (language tree-il))

(define forma
  '(maplexic cria objeto nome torre posicao (10 20)
             largura 30 altura 40 cor grama))

(display "ORIGINAL:")(newline)
(display "  ")(write forma)(newline)(newline)

(display "EXPANDIDA:")(newline)
(display "  ")(write (tree-il->scheme (macroexpand forma)))(newline)

ORIGINAL:
  (maplexic cria objeto nome torre posicao (10 20) largura 30 altura 40 cor grama)

EXPANDIDA:
  (construir (quote objeto) (quote ((nome . torre) (posicao 10 20) (largura . 30) (altura . 40) (cor . "#5bb04a"))))


In [9]:
; Validação (erro): falta campos
(maplexic cria objeto nome torre posicao (0 0) cor pedra)

misc-error: (#f ~A ~S (maplexic cria. campo faltante: largura) #f)

In [10]:
; Validação (erro): cor não existe
(maplexic cria objeto nome bandeira posicao (10 10) largura 10 altura 10 cor arco-iris)

syntax-error: (maplexic cor desconhecida (use textura ou hex "#rrggbb"): arco-iris ((line . 1) (column . 0)) (maplexic cria objeto nome bandeira posicao (10 10) largura 10 altura 10 cor arco-iris) #f)

In [11]:
(display "Adicionando objetos ao mapa-principal...\n")
(set! mapa-principal
  (maplexic adiciona (mata lago praia castelo estabulo heroi torre-mago) em mapa-principal))
(display mapa-principal)

Adicionando objetos ao mapa-principal...
(mapa mapa-principal 1000 1000 ((objeto mata (0 0) 150 1000 #1f5130) (objeto lago (250 650) 300 250 #3f7fe0) (objeto praia (250 550) 300 100 #d8c79a) (objeto castelo (650 600) 300 300 #8a8f99) (objeto estabulo-cheio (200 150) 180 120 #5bb04a) (objeto heroi (560 580) 30 40 #ff552f) (objeto torre-mago (720 350) 60 200 #8a4fd0)))

### Visualização (SVG)

In [12]:
;; ============================================================
;; Visualização em SVG: cada objeto vira um
;; quadrado da sua cor. A posição marca o canto inferior-esquerdo.
;; ============================================================
(define (mapa->svg registro arquivo)
  (let* ((nome    (symbol->string (acessa 'nome registro)))
         (largura (acessa 'largura registro))
         (altura  (acessa 'altura registro))
         (objetos (acessa 'objetos registro)))
    (when (file-exists? arquivo) (delete-file arquivo))
    (call-with-output-file arquivo
      (lambda (p)
        (define (linha . xs)
          (for-each (lambda (x) (display x p)) xs)
          (newline p))
        (linha "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 "
               largura " " altura "' width='640'>")
        (linha "  <rect width='" largura "' height='" altura
               "' fill='#0f1420' stroke='#3a4a66' stroke-width='2'/>")
        (linha "  <text x='10' y='28' fill='#e8eefc' font-family='monospace'"
               " font-size='28'>" nome "  " largura "x" altura "</text>")
        (for-each
         (lambda (obj)
           (let* ((on  (symbol->string (acessa 'nome obj)))
                  (pos (acessa 'posicao obj))
                  (x   (car pos))
                  (y   (cadr pos))
                  (w   (acessa 'largura obj))
                  (h   (acessa 'altura obj))
                  (css (acessa 'cor obj))
                  ;; SVG tem y para baixo; o topo do quadrado em coords de
                  ;; mapa é (y + h), então convertemos para o sistema do SVG
                  (sy  (- altura (+ y h))))
             (linha "  <rect x='" x "' y='" sy "' width='" w "' height='" h
                    "' fill='" css "' stroke='#0a0e16' stroke-width='2'/>")
             (linha "  <text x='" x "' y='" (- sy 6)
                    "' fill='#e8eefc' font-family='monospace' font-size='20'>"
                    on "</text>")))
         objetos)
        (linha "</svg>")))))

(display "funções de visualização criadas com sucesso")

funções de visualização criadas com sucesso

In [13]:
(mapa->svg mapa-principal "mapa-principal.svg")